# HYPERVIEW2 Compression Downstream Evaluation

Jeden notebook do pelnej walidacji downstream dla kompresji HYPERVIEW2:

1. przygotowuje repo, zaleznosci, Google Drive i dataset,
2. wykorzystuje zapisane rekonstrukcje lub opcjonalnie generuje nowe z checkpointow,
3. uruchamia regresory na oryginale i rekonstrukcjach,
4. zapisuje CSV/JSON z wynikami,
5. pokazuje interaktywne diagnostyki: score, targety, predykcje, pasma i widma.

Domyslny preset uzywa teraz HySpecNet-compatible normalizacji `reflectance_0_1` oraz diagnostycznego mapowania `230 -> 202 -> 230`, z checkpointem Mamby ladowanym natywnie bez adaptera kanalow.

Wyniki sa diagnostyka transferu na HYPERVIEW2, a nie reference-comparable metrykami HySpecNet-11k.

## 1. Repo

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/mhx1467/master-thesis-code.git'
REPO_DIR = Path('/content/hsi')
REPO_REF = 'main'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)

for module_name in list(sys.modules):
    if module_name == 'hsi_compression' or module_name.startswith('hsi_compression.'):
        del sys.modules[module_name]

os.chdir(REPO_DIR)
print('Repo:', Path.cwd())
print('Git:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

## 2. Zaleznosci

In [ ]:
# Main switch. Default mode evaluates already saved reconstructions from Drive and does
# not need mamba-ssm. Set RUN_RECONSTRUCTION=True only when you want this notebook to
# generate new Mamba reconstructions from checkpoints inside Colab.
RUN_RECONSTRUCTION = True
# Use prebuilt wheels instead of compiling causal-conv1d/mamba-ssm from source.
# This is the Colab reconstruction preset for Python 3.12 + CUDA 12 runtimes.
MAMBA_INSTALL_MODE = 'prebuilt_wheels' if RUN_RECONSTRUCTION else 'none'
INSTALL_MAMBA = RUN_RECONSTRUCTION and MAMBA_INSTALL_MODE != 'none'
INSTALL_OPTIONAL_BOOSTING = False
FORCE_REINSTALL_ENV = False

from pathlib import Path
import os
import subprocess
import sys

PIP = [sys.executable, '-m', 'pip']
marker_name = f'.hsi_compression_colab_env_v6_{MAMBA_INSTALL_MODE}'
ENV_MARKER = Path('/content') / marker_name

print(
    'Colab mode:',
    f'RUN_RECONSTRUCTION={RUN_RECONSTRUCTION}',
    f'MAMBA_INSTALL_MODE={MAMBA_INSTALL_MODE}',
    f'INSTALL_MAMBA={INSTALL_MAMBA}',
)


def run_install(cmd, *, required=True):
    print('Running:', ' '.join(cmd))
    result = subprocess.run(
        cmd,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout[-6000:])
    if result.returncode != 0:
        message = f'Command failed with exit code {result.returncode}: {" ".join(cmd)}'
        if required:
            raise RuntimeError(message)
        print('Optional install skipped:', message)
        return False
    return True


if FORCE_REINSTALL_ENV and ENV_MARKER.exists():
    ENV_MARKER.unlink()

if not ENV_MARKER.exists():
    base_install_cmds = [
        PIP + ['install', '-q', '--upgrade', 'pip', 'setuptools<82', 'wheel', 'packaging', 'pybind11', 'ninja'],
        PIP + [
            'install', '-q', '--upgrade', '--force-reinstall',
            'numpy==1.26.4',
            'pandas==2.2.2',
            'scipy>=1.12,<1.15',
            'scikit-learn>=1.6,<1.8',
        ],
        PIP + ['install', '-q', '-e', '.[downstream]', 'eotdl', 'tqdm', 'matplotlib', 'ipywidgets'],
    ]
    for cmd in base_install_cmds:
        run_install(cmd, required=True)

    if INSTALL_MAMBA:
        if MAMBA_INSTALL_MODE == 'prebuilt_wheels':
            run_install(PIP + [
                'install', '-q', '--force-reinstall',
                'torch==2.7.1',
                'torchvision==0.22.1',
                'torchaudio==2.7.1',
                '--index-url', 'https://download.pytorch.org/whl/cu126',
            ], required=True)
            run_install(PIP + [
                'install', '-q', '--force-reinstall', '--no-deps',
                'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1%2Bcu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
            ], required=True)
            run_install(PIP + [
                'install', '-q', '--force-reinstall', '--no-deps',
                'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1%2Bcu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
            ], required=True)
        elif MAMBA_INSTALL_MODE == 'pip_build':
            for cmd in [
                PIP + ['install', '--no-build-isolation', 'causal-conv1d>=1.4.0'],
                PIP + ['install', '--no-build-isolation', 'mamba-ssm>=2.3.1'],
            ]:
                run_install(cmd, required=True)
        else:
            raise ValueError(f'Unsupported MAMBA_INSTALL_MODE: {MAMBA_INSTALL_MODE}')

    if INSTALL_OPTIONAL_BOOSTING:
        run_install(PIP + ['install', '-q', 'lightgbm', 'catboost', 'xgboost'], required=True)

    ENV_MARKER.write_text('installed\n', encoding='utf-8')
    print('Dependencies installed. Colab runtime will restart once to reload NumPy ABI.')
    print('After reconnect, rerun from the repo cell; this cell will skip reinstalling.')
    os.kill(os.getpid(), 9)
else:
    print('Dependency marker exists, skipping reinstall:', ENV_MARKER)

import numpy as np
import torch

print('Python:', sys.version)
print('NumPy:', np.__version__)
print('Torch:', torch.__version__)
if INSTALL_MAMBA and not torch.__version__.startswith('2.7.'):
    if ENV_MARKER.exists():
        ENV_MARKER.unlink()
    raise RuntimeError(
        f'Mamba prebuilt wheels require Torch 2.7.x, but active Torch is {torch.__version__}. '
        'Restart the Colab runtime and rerun from the first cell so the v6 installer can reinstall Torch 2.7.1.'
    )
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
elif RUN_RECONSTRUCTION:
    raise RuntimeError('RUN_RECONSTRUCTION=True requires a GPU runtime.')

MAMBA_AVAILABLE = False
if INSTALL_MAMBA:
    try:
        from mamba_ssm import Mamba
        MAMBA_AVAILABLE = True
        print('mamba-ssm import: ok')
    except Exception as exc:
        raise RuntimeError(
            'mamba-ssm is required only for generating new Mamba reconstructions in Colab. '
            'Set RUN_RECONSTRUCTION=False to evaluate saved reconstructions.'
        ) from exc
else:
    print('Mamba install skipped; notebook will use saved reconstructions from Drive.')

## 3. Drive, dataset i katalogi wynikow

In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')

DRIVE_HSI = Path('/content/drive/MyDrive/hsi')
DRIVE_DATA_PARENT = DRIVE_HSI / 'data/hyperview2'
DRIVE_HV2_ROOT = DRIVE_DATA_PARENT / 'HYPERVIEW2'
DRIVE_DATA_ARCHIVE = DRIVE_DATA_PARENT / 'HYPERVIEW2_20260525.tar.gz'
DRIVE_CHECKPOINTS = DRIVE_HSI / 'checkpoints'
DRIVE_RECONS = DRIVE_HSI / 'reconstructions/hyperview2'
DRIVE_RESULTS = DRIVE_HSI / 'downstream_results/hyperview2_compression_hyspecnet202_resample'

for path in [DRIVE_DATA_PARENT, DRIVE_CHECKPOINTS, DRIVE_RECONS, DRIVE_RESULTS]:
    path.mkdir(parents=True, exist_ok=True)

LOCAL_DATA_PARENT = Path('/content/data/hyperview2')
LOCAL_DATA_PARENT.mkdir(parents=True, exist_ok=True)


def is_hyperview2_root(path: Path) -> bool:
    required = [
        path / 'train_gt.csv',
        path / 'submission.csv',
        path / 'train/hsi_satellite',
        path / 'test/hsi_satellite',
    ]
    return all(item.exists() for item in required)


def find_hyperview2_root(search_root: Path) -> Path | None:
    if is_hyperview2_root(search_root):
        return search_root
    for candidate in sorted(search_root.rglob('HYPERVIEW2')) if search_root.exists() else []:
        if is_hyperview2_root(candidate):
            return candidate
    return None


HV2_ROOT = find_hyperview2_root(DRIVE_DATA_PARENT)
if HV2_ROOT is None and DRIVE_DATA_ARCHIVE.exists():
    print('Extracting HYPERVIEW2 archive from Drive:', DRIVE_DATA_ARCHIVE)
    subprocess.run(['tar', '-xzf', str(DRIVE_DATA_ARCHIVE), '-C', str(DRIVE_DATA_PARENT)], check=True)
    HV2_ROOT = find_hyperview2_root(DRIVE_DATA_PARENT)

if HV2_ROOT is None:
    print('Dataset not found on Drive. Downloading with EOTDL to local Colab storage...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'eotdl'], check=True)
    subprocess.run(['eotdl', 'datasets', 'get', 'HYPERVIEW2', '-o', str(LOCAL_DATA_PARENT)], check=True)
    HV2_ROOT = find_hyperview2_root(LOCAL_DATA_PARENT)

if HV2_ROOT is None or not is_hyperview2_root(HV2_ROOT):
    raise FileNotFoundError('Could not resolve canonical HYPERVIEW2 root.')

print('HV2_ROOT:', HV2_ROOT)
for rel in ['train/hsi_satellite', 'train/hsi_airborne', 'train/msi_satellite', 'test/hsi_satellite', 'test/msi_satellite']:
    directory = HV2_ROOT / rel
    count = len(list(directory.glob('*.npz'))) if directory.exists() else 0
    print(f'{rel:24s} {count:5d} npz files')
print('Checkpoints:', DRIVE_CHECKPOINTS)
print('Reconstructions:', DRIVE_RECONS)
print('Results:', DRIVE_RESULTS)

## 4. Konfiguracja eksperymentu

In [ ]:
from pathlib import Path
from typing import Any
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

from hsi_compression.downstream import HYPERVIEW2_TARGET_COLUMNS, build_hyperview2_samples, split_samples
from hsi_compression.downstream.hyperview2_compression_eval import (
    CompressionCheckpoint,
    best_by_variant_mode,
    compute_per_band_diagnostics,
    discover_recon_roots,
    evaluate_downstream_regressors,
    infer_recon_input_normalization,
    load_cube_and_value_mask,
    normalize_original_cube,
    prediction_metrics_by_target,
    prediction_shift_decomposition,
    read_reconstruction_summary,
    reconstruct_affine_calibrated_reconstruction,
    reconstruct_checkpoint,
    reconstruct_spectral_resample_passthrough,
    safe_sample_stem,
    sample_path,
    save_downstream_artifacts,
)

plt.rcParams['figure.dpi'] = 130
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
TARGETS = list(HYPERVIEW2_TARGET_COLUMNS)

# quick: fast checkpoint triage, subset diagnostic, not final-report comparable.
# balanced: all train/val samples, two fast tree models, still no slow HGB.
# full: all samples and the previous 3-regressor comparison table.
EVAL_PRESET = 'quick'
if EVAL_PRESET not in {'quick', 'balanced', 'full'}:
    raise ValueError("EVAL_PRESET must be one of: quick, balanced, full")

MODALITY = 'prisma'
FEATURE_SET = 'mean_std_derivatives'
ORIGINAL_FEATURE_NORMALIZATION = 'reflectance_0_1'
CHECKPOINT_NORMALIZATION_FALLBACK = 'reflectance_0_1'
VAL_FRACTION = 0.2
SEED = 42
N_JOBS = -1
FEATURE_DEVICE = DEVICE          # PyTorch feature extraction runs on GPU when available
FEATURE_BATCH_SIZE = 128
# Colab/Jupyter may emit DataLoader multiprocessing cleanup assertions with workers>0.
# Keep notebook DataLoaders single-process; tensor work still runs on FEATURE_DEVICE/GPU.
COLAB_DATALOADER_NUM_WORKERS = 0
FEATURE_NUM_WORKERS = COLAB_DATALOADER_NUM_WORKERS
VERBOSE_EVAL = True

if EVAL_PRESET == 'quick':
    MODEL_NAMES = ['extra_trees']
    MAX_TRAIN_SAMPLES = 600
    MAX_VAL_SAMPLES = 150
    FORCE_RECONSTRUCTION = False
    ADD_LOSSLESS_PASSTHROUGH = False
    ADD_AFFINE_CALIBRATED_RECONS = False
    MAX_PER_BAND_SAMPLES = 80
elif EVAL_PRESET == 'balanced':
    MODEL_NAMES = ['extra_trees', 'random_forest']
    MAX_TRAIN_SAMPLES = None
    MAX_VAL_SAMPLES = None
    FORCE_RECONSTRUCTION = False
    ADD_LOSSLESS_PASSTHROUGH = False
    ADD_AFFINE_CALIBRATED_RECONS = False
    MAX_PER_BAND_SAMPLES = 300
else:
    MODEL_NAMES = ['hist_gradient_boosting', 'extra_trees', 'random_forest']
    MAX_TRAIN_SAMPLES = None
    MAX_VAL_SAMPLES = None
    FORCE_RECONSTRUCTION = False
    ADD_LOSSLESS_PASSTHROUGH = True
    ADD_AFFINE_CALIBRATED_RECONS = True
    MAX_PER_BAND_SAMPLES = None

# Use a preset-specific output directory so quick subset diagnostics do not overwrite full results.
DRIVE_RESULTS = DRIVE_HSI / f'downstream_results/hyperview2_compression_hyspecnet202_resample_{EVAL_PRESET}'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

AUTO_DISCOVER_RECONS = False
RECOMPUTE_REGRESSORS = True     # train/evaluate regressors again; set False only to load cached CSV/JSON
ADD_SPECTRAL_RESAMPLE_PASSTHROUGH = True  # original_230 -> resample_202 -> inverse_230, no Mamba
SPECTRAL_RESAMPLE_PASSTHROUGH_VARIANT = 'hyperview2_spectral_resample_passthrough_hyspecnet202_to_230'
SPECTRAL_RESAMPLE_BATCH_SIZE = 32
SPECTRAL_RESAMPLE_NUM_WORKERS = COLAB_DATALOADER_NUM_WORKERS
AFFINE_CALIBRATION_SOURCE_VARIANTS = [
    'hyperview2_mamba_k4_hv2_finetune_hyspecnet202_to_230',
]
AFFINE_CALIBRATION_SUFFIX = '_train_affine_calibrated'
AFFINE_CALIBRATION_BATCH_SIZE = 32
AFFINE_CALIBRATION_NUM_WORKERS = COLAB_DATALOADER_NUM_WORKERS
VARIANTS_TO_EVALUATE: list[str] | str = 'auto'  # or list of variant names

ENABLE_HV2_FINETUNE_CKPT = True
ENABLE_HYSPECNET_SOURCE_CKPT = EVAL_PRESET == 'full'
ENABLE_HV2_LAST_CKPT = False

# HYPERVIEW2 fine-tuned checkpoint produced by notebooks/hyperview2_mamba_finetune_colab.ipynb.
HV2_FINETUNE_BEST_CKPT = DRIVE_CHECKPOINTS / 'hyperview2_prisma_hyspecnet202_mamba_k4_spatial_rd_lambda_0_001_spectral_feature_ft_best.pt'
HV2_FINETUNE_LAST_CKPT = DRIVE_CHECKPOINTS / 'hyperview2_prisma_hyspecnet202_mamba_k4_spatial_rd_lambda_0_001_spectral_feature_ft_last.pt'

# Previous HySpecNet-trained spectral-feature checkpoint, useful for full comparison only.
HYSPECNET_SPECTRAL_FEATURE_CKPT = DRIVE_CHECKPOINTS / 'hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_001_spectral_feature_ft_best.pt'

CHECKPOINTS = []
if ENABLE_HV2_FINETUNE_CKPT:
    if HV2_FINETUNE_BEST_CKPT.exists():
        CHECKPOINTS.append(CompressionCheckpoint(
            name='mamba_k4_hv2_finetune_hyspecnet202_resample_best',
            path=HV2_FINETUNE_BEST_CKPT,
            variant_name='hyperview2_mamba_k4_hv2_finetune_hyspecnet202_to_230',
            compression_normalization='reflectance_0_1',
            recon_feature_normalization='none',
            batch_size=1,
            num_workers=COLAB_DATALOADER_NUM_WORKERS,
            use_bitstream=True,
            allow_in_channel_adapter=False,
            spectral_mapping='hyspecnet_202_approx',
        ))
    else:
        print('HYPERVIEW2 fine-tuned best checkpoint missing:', HV2_FINETUNE_BEST_CKPT)

if ENABLE_HV2_LAST_CKPT:
    if HV2_FINETUNE_LAST_CKPT.exists():
        CHECKPOINTS.append(CompressionCheckpoint(
            name='mamba_k4_hv2_finetune_hyspecnet202_resample_last',
            path=HV2_FINETUNE_LAST_CKPT,
            variant_name='hyperview2_mamba_k4_hv2_finetune_last_hyspecnet202_to_230',
            compression_normalization='reflectance_0_1',
            recon_feature_normalization='none',
            batch_size=1,
            num_workers=COLAB_DATALOADER_NUM_WORKERS,
            use_bitstream=True,
            allow_in_channel_adapter=False,
            spectral_mapping='hyspecnet_202_approx',
        ))
    else:
        print('HYPERVIEW2 fine-tuned last checkpoint missing:', HV2_FINETUNE_LAST_CKPT)

if ENABLE_HYSPECNET_SOURCE_CKPT:
    if HYSPECNET_SPECTRAL_FEATURE_CKPT.exists():
        CHECKPOINTS.append(CompressionCheckpoint(
            name='mamba_k4_spectral_feature_ft_hyspecnet202_resample',
            path=HYSPECNET_SPECTRAL_FEATURE_CKPT,
            variant_name='hyperview2_mamba_k4_spectral_feature_ft_epoch16_hyspecnet202_to_230',
            compression_normalization='reflectance_0_1',
            recon_feature_normalization='none',
            batch_size=1,
            num_workers=COLAB_DATALOADER_NUM_WORKERS,
            use_bitstream=True,
            allow_in_channel_adapter=False,
            spectral_mapping='hyspecnet_202_approx',
        ))
    else:
        print('HySpecNet source checkpoint missing:', HYSPECNET_SPECTRAL_FEATURE_CKPT)

print('Preset:', EVAL_PRESET)
print('Results dir:', DRIVE_RESULTS)
print('Device:', DEVICE)
print('Feature extraction device:', FEATURE_DEVICE)
print('Regressor backend: scikit-learn CPU, parallelized with N_JOBS=', N_JOBS)
print('Models:', MODEL_NAMES)
print('Sample limits:', 'train=', MAX_TRAIN_SAMPLES, 'val=', MAX_VAL_SAMPLES)
print('Checkpoints to reconstruct/check:', [ckpt.name for ckpt in CHECKPOINTS])

## 5. Rekonstrukcje: wygeneruj lub uzyj zapisanych

In [ ]:
reconstruction_summaries: dict[str, Any] = {}
RECON_ROOTS: dict[str, Path] = {}
RECON_INPUT_NORMALIZATIONS: dict[str, str] = {}
RECON_FEATURE_NORMALIZATIONS: dict[str, str] = {}

if AUTO_DISCOVER_RECONS:
    for variant, root in discover_recon_roots(DRIVE_RECONS).items():
        summary = read_reconstruction_summary(root)
        RECON_ROOTS[variant] = root
        RECON_INPUT_NORMALIZATIONS[variant] = infer_recon_input_normalization(variant, summary)
        RECON_FEATURE_NORMALIZATIONS[variant] = str((summary or {}).get('recon_feature_normalization', 'none'))
        if summary:
            reconstruction_summaries[variant] = summary

if ADD_LOSSLESS_PASSTHROUGH:
    variant = 'lossless_passthrough_exact'
    RECON_ROOTS[variant] = HV2_ROOT
    RECON_INPUT_NORMALIZATIONS[variant] = ORIGINAL_FEATURE_NORMALIZATION
    RECON_FEATURE_NORMALIZATIONS[variant] = ORIGINAL_FEATURE_NORMALIZATION
    reconstruction_summaries[variant] = {
        'variant': variant,
        'recon_root': str(HV2_ROOT),
        'note': 'Exact reconstruction sanity upper bound; no codec bitstream generated in this notebook.',
    }

if ADD_SPECTRAL_RESAMPLE_PASSTHROUGH:
    variant = SPECTRAL_RESAMPLE_PASSTHROUGH_VARIANT
    expected_root = DRIVE_RECONS / variant / 'HYPERVIEW2'
    if expected_root.exists() and not FORCE_RECONSTRUCTION:
        print('Using existing spectral-resample passthrough:', variant, '->', expected_root)
        summary = read_reconstruction_summary(expected_root) or {}
    else:
        expected_root, summary = reconstruct_spectral_resample_passthrough(
            source_root=HV2_ROOT,
            recon_parent=DRIVE_RECONS,
            device=DEVICE,
            variant_name=variant,
            modality=MODALITY,
            normalization=ORIGINAL_FEATURE_NORMALIZATION,
            spectral_mapping_name='hyspecnet_202_approx',
            batch_size=SPECTRAL_RESAMPLE_BATCH_SIZE,
            num_workers=SPECTRAL_RESAMPLE_NUM_WORKERS,
            split='train',
        )
    RECON_ROOTS[variant] = expected_root
    RECON_INPUT_NORMALIZATIONS[variant] = str(summary.get('input_normalization', ORIGINAL_FEATURE_NORMALIZATION))
    RECON_FEATURE_NORMALIZATIONS[variant] = str(summary.get('recon_feature_normalization', 'none'))
    reconstruction_summaries[variant] = summary

for checkpoint in CHECKPOINTS:
    variant = checkpoint.variant_name or f'{checkpoint.name}_input_{checkpoint.compression_normalization}'
    expected_root = DRIVE_RECONS / variant / 'HYPERVIEW2'
    if expected_root.exists() and not FORCE_RECONSTRUCTION:
        print('Using existing reconstruction:', variant, '->', expected_root)
        summary = read_reconstruction_summary(expected_root) or {}
    elif RUN_RECONSTRUCTION:
        expected_root, summary = reconstruct_checkpoint(
            checkpoint,
            source_root=HV2_ROOT,
            recon_parent=DRIVE_RECONS,
            device=DEVICE,
            checkpoint_normalization_fallback=CHECKPOINT_NORMALIZATION_FALLBACK,
            split='train',
        )
    else:
        print('Skipping checkpoint reconstruction because RUN_RECONSTRUCTION=False and root is missing:', variant)
        continue
    RECON_ROOTS[variant] = expected_root
    RECON_INPUT_NORMALIZATIONS[variant] = str(summary.get('input_normalization', checkpoint.compression_normalization))
    RECON_FEATURE_NORMALIZATIONS[variant] = checkpoint.recon_feature_normalization
    reconstruction_summaries[variant] = summary

if ADD_AFFINE_CALIBRATED_RECONS:
    original_samples_for_calibration = build_hyperview2_samples(HV2_ROOT, modality=MODALITY, split='train')
    calibration_train_samples, _ = split_samples(
        original_samples_for_calibration,
        val_fraction=VAL_FRACTION,
        seed=SEED,
    )
    calibration_ids = [str(sample.sample_id) for sample in calibration_train_samples]
    print('Affine calibration train samples:', len(calibration_ids))
    for source_variant in AFFINE_CALIBRATION_SOURCE_VARIANTS:
        if source_variant not in RECON_ROOTS:
            print('Skipping affine calibration because source variant is missing:', source_variant)
            continue
        variant = f'{source_variant}{AFFINE_CALIBRATION_SUFFIX}'
        expected_root = DRIVE_RECONS / variant / 'HYPERVIEW2'
        if expected_root.exists() and not FORCE_RECONSTRUCTION:
            print('Using existing affine-calibrated reconstruction:', variant, '->', expected_root)
            summary = read_reconstruction_summary(expected_root) or {}
        else:
            expected_root, summary = reconstruct_affine_calibrated_reconstruction(
                original_root=HV2_ROOT,
                recon_root=RECON_ROOTS[source_variant],
                recon_parent=DRIVE_RECONS,
                variant_name=variant,
                calibration_sample_ids=calibration_ids,
                device=DEVICE,
                source_variant=source_variant,
                modality=MODALITY,
                original_normalization=ORIGINAL_FEATURE_NORMALIZATION,
                recon_normalization=RECON_FEATURE_NORMALIZATIONS.get(source_variant, 'none'),
                batch_size=AFFINE_CALIBRATION_BATCH_SIZE,
                num_workers=AFFINE_CALIBRATION_NUM_WORKERS,
                split='train',
            )
        RECON_ROOTS[variant] = expected_root
        RECON_INPUT_NORMALIZATIONS[variant] = ORIGINAL_FEATURE_NORMALIZATION
        RECON_FEATURE_NORMALIZATIONS[variant] = str(summary.get('recon_feature_normalization', 'none'))
        reconstruction_summaries[variant] = summary

if VARIANTS_TO_EVALUATE != 'auto':
    selected = set(VARIANTS_TO_EVALUATE)
    RECON_ROOTS = {name: root for name, root in RECON_ROOTS.items() if name in selected}
    RECON_INPUT_NORMALIZATIONS = {name: RECON_INPUT_NORMALIZATIONS[name] for name in RECON_ROOTS}
    RECON_FEATURE_NORMALIZATIONS = {name: RECON_FEATURE_NORMALIZATIONS[name] for name in RECON_ROOTS}

print('Variants used for downstream evaluation:')
for name, root in RECON_ROOTS.items():
    print(
        ' ', name,
        '| root=', root,
        '| input_norm=', RECON_INPUT_NORMALIZATIONS.get(name),
        '| feature_norm=', RECON_FEATURE_NORMALIZATIONS.get(name),
    )
if not RECON_ROOTS:
    print('No reconstruction variants found. Add CHECKPOINTS or place reconstructions under Drive.')

## 6. Ewaluacja regresorow

In [ ]:
cached_paths = {
    'summary_csv': DRIVE_RESULTS / 'compression_downstream_summary.csv',
    'predictions_csv': DRIVE_RESULTS / 'compression_downstream_predictions.csv',
    'metrics_json': DRIVE_RESULTS / 'compression_downstream_metrics.json',
}
cache_available = all(path.exists() for path in cached_paths.values())

if RECOMPUTE_REGRESSORS or not cache_available:
    print('Training/evaluating downstream regressors from scratch...')
    print('Preset:', EVAL_PRESET)
    print('Models:', MODEL_NAMES)
    print('Reconstruction variants:', list(RECON_ROOTS))
    print('Sample limits:', 'train=', MAX_TRAIN_SAMPLES, 'val=', MAX_VAL_SAMPLES)
    results_df, predictions_df, metrics_payload = evaluate_downstream_regressors(
        hv2_root=HV2_ROOT,
        recon_roots=RECON_ROOTS,
        recon_feature_normalizations=RECON_FEATURE_NORMALIZATIONS,
        model_names=MODEL_NAMES,
        modality=MODALITY,
        feature_set=FEATURE_SET,
        original_feature_normalization=ORIGINAL_FEATURE_NORMALIZATION,
        val_fraction=VAL_FRACTION,
        seed=SEED,
        n_jobs=N_JOBS,
        feature_device=FEATURE_DEVICE,
        feature_batch_size=FEATURE_BATCH_SIZE,
        feature_num_workers=FEATURE_NUM_WORKERS,
        max_train_samples=MAX_TRAIN_SAMPLES,
        max_val_samples=MAX_VAL_SAMPLES,
        verbose=VERBOSE_EVAL,
    )
    metrics_payload['reconstruction_summaries'] = reconstruction_summaries
    metrics_payload['protocol']['eval_preset'] = EVAL_PRESET
    metrics_payload['protocol']['recon_input_normalizations'] = RECON_INPUT_NORMALIZATIONS
    metrics_payload['protocol']['checkpoints'] = [ckpt.to_record() for ckpt in CHECKPOINTS]

    paths = save_downstream_artifacts(DRIVE_RESULTS, results_df, predictions_df, metrics_payload)
    for label, path in paths.items():
        print(label, '->', path)
else:
    print('Loading cached downstream results. Set RECOMPUTE_REGRESSORS=True to retrain.')
    results_df = pd.read_csv(cached_paths['summary_csv'])
    predictions_df = pd.read_csv(cached_paths['predictions_csv'])
    metrics_payload = json.loads(cached_paths['metrics_json'].read_text())

best_df = best_by_variant_mode(results_df)
display(best_df[['variant', 'mode', 'model', 'hyperview_score', 'mean_mse', 'mean_mae', 'fit_time_sec', 'predict_time_sec']])

## 7. Szybkie podsumowanie wynikow

In [ ]:
best_df = best_by_variant_mode(results_df)
original_score = best_df.loc[best_df['variant'].eq('original'), 'hyperview_score'].min()
summary = best_df.copy()
summary['ratio_to_original_best'] = summary['hyperview_score'] / original_score
summary_path = DRIVE_RESULTS / 'best_scores.csv'
summary.to_csv(summary_path, index=False)
print('Saved:', summary_path)
display(summary[['variant', 'mode', 'model', 'hyperview_score', 'ratio_to_original_best']])

plot_df = summary.sort_values(['variant', 'mode']).copy()
labels = plot_df['variant'] + '\n' + plot_df['mode'] + '\n' + plot_df['model']
fig, ax = plt.subplots(figsize=(max(9, 0.55 * len(plot_df)), 4.5))
colors = ['#4c78a8' if row.variant == 'original' else '#f58518' for row in plot_df.itertuples()]
ax.bar(np.arange(len(plot_df)), plot_df['hyperview_score'], color=colors)
ax.axhline(1.0, color='black', linewidth=1, linestyle='--', label='dummy mean')
ax.set_ylabel('Hyperview score (lower is better)')
ax.set_xticks(np.arange(len(plot_df)))
ax.set_xticklabels(labels, rotation=60, ha='right')
ax.grid(True, axis='y', alpha=0.25)
ax.legend(loc='upper right')
fig.tight_layout()
out = DRIVE_RESULTS / 'best_downstream_scores.png'
fig.savefig(out, bbox_inches='tight')
print('Saved:', out)
plt.show()

## 8. Targety i przesuniecie predykcji

In [ ]:
target_metrics = prediction_metrics_by_target(predictions_df)
shift_df = prediction_shift_decomposition(predictions_df)

target_metrics_path = DRIVE_RESULTS / 'prediction_metrics_by_target.csv'
shift_path = DRIVE_RESULTS / 'prediction_shift_decomposition.csv'
target_metrics.to_csv(target_metrics_path, index=False)
shift_df.to_csv(shift_path, index=False)
print('Saved:', target_metrics_path)
print('Saved:', shift_path)

best_keys = best_df[['variant', 'mode', 'model']]
focus_targets = target_metrics.merge(best_keys, on=['variant', 'mode', 'model'], how='inner')
display(focus_targets.sort_values(['variant', 'mode', 'relative_mse']).reset_index(drop=True))

if not focus_targets.empty:
    rows = list(best_df.itertuples(index=False))
    fig, axes = plt.subplots(len(rows), 1, figsize=(9, max(3, 2.2 * len(rows))), sharex=True)
    if len(rows) == 1:
        axes = [axes]
    for ax, row in zip(axes, rows):
        vals = [getattr(row, f'{target}_relative_mse', np.nan) for target in TARGETS]
        ax.bar(TARGETS, vals, color='#72b7b2')
        ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
        ax.set_ylabel('rel. MSE')
        ax.set_title(f'{row.variant} | {row.mode} | {row.model} | score={row.hyperview_score:.3f}')
        ax.grid(True, axis='y', alpha=0.25)
    axes[-1].set_xlabel('Target')
    fig.tight_layout()
    out = DRIVE_RESULTS / 'per_target_relative_mse_best_models.png'
    fig.savefig(out, bbox_inches='tight')
    print('Saved:', out)
    plt.show()

display(shift_df.sort_values(['variant', 'model', 'extra_mse'], ascending=[True, True, False]).reset_index(drop=True))

## 9. Diagnostyka pasm: MAE, bias i najgorsze bandy

In [ ]:
val_ids = [str(item) for item in metrics_payload['protocol']['val_sample_ids']]
per_band_tables = []
sample_error_tables = []

for variant, root in RECON_ROOTS.items():
    original_norm = RECON_INPUT_NORMALIZATIONS.get(variant, 'none')
    per_band, sample_errors = compute_per_band_diagnostics(
        original_root=HV2_ROOT,
        recon_root=root,
        sample_ids=val_ids,
        original_normalization=original_norm,
        max_samples=MAX_PER_BAND_SAMPLES,
    )
    per_band['variant'] = variant
    sample_errors['variant'] = variant
    per_band_tables.append(per_band)
    sample_error_tables.append(sample_errors)

per_band_df = pd.concat(per_band_tables, ignore_index=True) if per_band_tables else pd.DataFrame()
sample_errors_df = pd.concat(sample_error_tables, ignore_index=True) if sample_error_tables else pd.DataFrame()
per_band_path = DRIVE_RESULTS / 'per_band_diagnostics.csv'
sample_errors_path = DRIVE_RESULTS / 'sample_reconstruction_errors.csv'
per_band_df.to_csv(per_band_path, index=False)
sample_errors_df.to_csv(sample_errors_path, index=False)
print('Saved:', per_band_path)
print('Saved:', sample_errors_path)

display(per_band_df.groupby(['variant', 'original_normalization'])[['mae', 'rmse', 'bias']].mean().reset_index())

top_bands = (
    per_band_df.sort_values(['variant', 'mae'], ascending=[True, False])
    .groupby('variant')
    .head(12)
    [['variant', 'band', 'mae', 'rmse', 'bias', 'orig_mean', 'recon_mean']]
)
display(top_bands.reset_index(drop=True))

if not per_band_df.empty:
    fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
    for variant, group in per_band_df.groupby('variant'):
        axes[0].plot(group['band'], group['mae'], label=variant)
        axes[1].plot(group['band'], group['bias'], label=variant)
    axes[0].set_ylabel('MAE')
    axes[0].set_title('Per-band reconstruction error')
    axes[1].set_ylabel('Bias: reconstruction - reference')
    axes[1].set_xlabel('Band index')
    for ax in axes:
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)
    fig.tight_layout()
    out = DRIVE_RESULTS / 'per_band_error_curves.png'
    fig.savefig(out, bbox_inches='tight')
    print('Saved:', out)
    plt.show()

## 10. Interaktywna analiza widm i obrazow pasm

In [ ]:
import ipywidgets as widgets
from IPython.display import display


def mean_spectrum(cube: np.ndarray, mask: np.ndarray) -> np.ndarray:
    counts = mask.sum(axis=(1, 2)).astype(np.float32)
    values = (cube * mask).sum(axis=(1, 2)).astype(np.float32)
    out = np.full(cube.shape[0], np.nan, dtype=np.float32)
    np.divide(values, counts, out=out, where=counts > 0)
    return out


def load_original_recon_pair(variant: str, sample_id: str):
    root = RECON_ROOTS[variant]
    original_norm = RECON_INPUT_NORMALIZATIONS.get(variant, 'none')
    orig, orig_mask = load_cube_and_value_mask(sample_path(HV2_ROOT, sample_id))
    recon, recon_mask = load_cube_and_value_mask(sample_path(root, sample_id))
    orig = normalize_original_cube(orig, orig_mask, original_norm)
    c = min(orig.shape[0], recon.shape[0])
    h = min(orig.shape[-2], recon.shape[-2])
    w = min(orig.shape[-1], recon.shape[-1])
    orig = orig[:c, :h, :w]
    recon = recon[:c, :h, :w]
    valid = orig_mask[:c, :h, :w] & recon_mask[:c, :h, :w]
    return orig, recon, valid, original_norm


variant_widget = widgets.Dropdown(options=list(RECON_ROOTS), description='variant')
sample_widget = widgets.Dropdown(options=val_ids, description='sample')
band_widget = widgets.IntSlider(value=0, min=0, max=229, step=1, description='band')


def show_reconstruction(variant: str, sample_id: str, band: int):
    orig, recon, valid, original_norm = load_original_recon_pair(variant, sample_id)
    band = int(np.clip(band, 0, orig.shape[0] - 1))
    orig_spec = mean_spectrum(orig, valid)
    recon_spec = mean_spectrum(recon, valid)
    diff_spec = recon_spec - orig_spec
    valid_band = valid[band]
    mae = float(np.abs((recon - orig)[valid]).mean()) if valid.any() else np.nan
    rmse = float(np.sqrt(((recon - orig)[valid] ** 2).mean())) if valid.any() else np.nan

    print(f'variant={variant} | sample={sample_id} | original_norm={original_norm} | MAE={mae:.6f} | RMSE={rmse:.6f}')
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    x = np.arange(orig.shape[0])
    axes[0, 0].plot(x, orig_spec, label='original/reference', linewidth=1.4)
    axes[0, 0].plot(x, recon_spec, label='reconstruction', linewidth=1.2)
    axes[0, 0].axvline(band, color='black', linestyle='--', linewidth=0.8)
    axes[0, 0].set_title('Mean spectrum')
    axes[0, 0].set_xlabel('Band')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.25)

    axes[1, 0].plot(x, diff_spec, color='#e45756')
    axes[1, 0].axhline(0.0, color='black', linewidth=0.8)
    axes[1, 0].axvline(band, color='black', linestyle='--', linewidth=0.8)
    axes[1, 0].set_title('Mean spectral error')
    axes[1, 0].set_xlabel('Band')
    axes[1, 0].grid(True, alpha=0.25)

    vmin = float(np.nanpercentile(orig[band][valid_band], 1)) if valid_band.any() else None
    vmax = float(np.nanpercentile(orig[band][valid_band], 99)) if valid_band.any() else None
    axes[0, 1].imshow(orig[band], cmap='viridis', vmin=vmin, vmax=vmax)
    axes[0, 1].set_title(f'Original/reference band {band}')
    axes[0, 1].axis('off')

    err = recon[band] - orig[band]
    lim = float(np.nanpercentile(np.abs(err[valid_band]), 99)) if valid_band.any() else 1.0
    axes[1, 1].imshow(err, cmap='coolwarm', vmin=-lim, vmax=lim)
    axes[1, 1].set_title(f'Reconstruction error band {band}')
    axes[1, 1].axis('off')
    fig.tight_layout()
    plt.show()


ui = widgets.VBox([variant_widget, sample_widget, band_widget])
out = widgets.interactive_output(
    show_reconstruction,
    {'variant': variant_widget, 'sample_id': sample_widget, 'band': band_widget},
)
display(ui, out)

## 11. Pliki wyjsciowe

In [ ]:
print('Results directory:', DRIVE_RESULTS)
for path in sorted(DRIVE_RESULTS.glob('*')):
    if path.is_file():
        print(path.name, f'{path.stat().st_size / 1024:.1f} KiB')